# 05 - Error analysis

Uses the predictions of the best transformer on the clean test set (saved by notebook 04).
Goal: find out *why* the model fails, not just how often.

In [ ]:
import sys
import collections

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append("../src")
from preprocess import LABELS
from metrics import get_scores, print_report

pd.set_option("display.max_colwidth", 200)
preds = pd.read_csv("../results/best_predictions_test_clean.csv")
train = pd.read_csv("../data/processed/train.csv")
preds["correct"] = preds["label"] == preds["pred"]
print_report(preds["label"], preds["pred"])

## Which mistakes happen most

In [ ]:
errors = preds[~preds["correct"]]
print(f"{len(errors)} errors out of {len(preds)} ({len(errors) / len(preds):.1%})")
errors.groupby(["label", "pred"]).size().sort_values(ascending=False).rename("count").to_frame()

## Error type 1: unknown words

Do tweets with more words never seen in training get more errors? Spelling variation and new slang
would show up here.

In [ ]:
train_words = set(" ".join(train["text"]).split())


def unseen_ratio(text):
    words = text.split()
    return sum(w not in train_words for w in words) / max(len(words), 1)


preds["unseen_ratio"] = preds["text"].apply(unseen_ratio)
preds["unseen_bin"] = pd.cut(preds["unseen_ratio"], [-0.01, 0.05, 0.15, 0.3, 1.0],
                             labels=["0-5%", "5-15%", "15-30%", "30%+"])
by_unseen = preds.groupby("unseen_bin", observed=True).agg(tweets=("correct", "size"), accuracy=("correct", "mean"))
by_unseen["macro_f1"] = [get_scores(g["label"], g["pred"])["macro_f1"]
                         for _, g in preds.groupby("unseen_bin", observed=True)]
by_unseen.round(3)

## Error type 2: tweet length

In [ ]:
preds["n_words"] = preds["text"].str.split().str.len()
preds["len_bin"] = pd.cut(preds["n_words"], [0, 8, 15, 25, 100], labels=["1-8", "9-15", "16-25", "26+"])
preds.groupby("len_bin", observed=True)["correct"].agg(["size", "mean"]).round(3)

## Error type 3: English vs Pidgin heavy tweets

In [ ]:
# a small list of very common pidgin function words, used as a rough "how pidgin is this tweet" score
PIDGIN_WORDS = {"dey", "na", "wey", "una", "dem", "don", "wetin", "abeg", "sef", "sha", "oya", "comot",
                "pikin", "wahala", "sabi", "chop", "am", "e", "go", "make", "no", "fit", "say", "ehn", "o"}


def pidgin_ratio(text):
    words = text.split()
    return sum(w in PIDGIN_WORDS for w in words) / max(len(words), 1)


preds["pidgin_ratio"] = preds["text"].apply(pidgin_ratio)
preds["pidgin_bin"] = pd.cut(preds["pidgin_ratio"], [-0.01, 0.05, 0.15, 0.3, 1.0],
                             labels=["mostly english", "light pidgin", "pidgin", "heavy pidgin"])
preds.groupby("pidgin_bin", observed=True)["correct"].agg(["size", "mean"]).round(3)

## Examples to read

I read these myself and sort them into reasons (sarcasm, mixed feelings, label I disagree with,
slang, neutral confused with negative, etc.).

In [ ]:
print("=== examples the model got right ===")
preds[preds["correct"]].sample(8, random_state=1)[["text", "label", "pred"]]

In [ ]:
print("=== neutral tweets (true label) the model got wrong ===")
errors[errors["label"] == "neutral"].sample(10, random_state=1)[["text", "label", "pred"]]

In [ ]:
print("=== positive predicted as negative ===")
errors[(errors["label"] == "positive") & (errors["pred"] == "negative")].sample(10, random_state=1)[["text", "label", "pred"]]

In [ ]:
print("=== negative predicted as positive ===")
errors[(errors["label"] == "negative") & (errors["pred"] == "positive")].sample(10, random_state=1)[["text", "label", "pred"]]

## My manual categories

I go through a sample of 50 errors and put each one in a category. (Filled in by hand, see the table
below.)

In [ ]:
sample = errors.sample(50, random_state=7)[["text", "label", "pred"]].reset_index(drop=True)
sample.to_csv("../results/error_sample_to_label.csv", index=False)
sample